# Boolean Network Analysis Workflow Example
This notebook demonstrate the workflow to simulate, extend and analyze a Boolean network through Probabilistic Boolean Network (PBN) approach.

In [ ]:
import sys
import KGBN

## Load a Boolean Network

Below is a synthetic model that describes the EGFR-RAS-MAPK pathway, and the DNA-damage–p53 apoptotic checkpoint.


In [ ]:
network_string = """
EGF = EGF
EGFR = EGF
DNA_DAMAGE  = DNA_DAMAGE
KRAS   = EGFR
RAF1   = KRAS
MAP2K1 = RAF1
MAPK1  = MAP2K1
PIK3CA = EGFR | MAPK1
PTEN   = !PIK3CA
AKT1   = PIK3CA & !PTEN
MDM2   = AKT1
TP53   = DNA_DAMAGE & !MDM2
CASP3  = TP53 & !AKT1
CDK2   = (MAPK1 | AKT1) & !TP53
"""
network = KGBN.load_network_from_string(network_string)

An example of experimental data is provided in `files/experiments_example.csv`.  
It includes the experimental conditions like stimulate EGFR and inhibit PDPK1, and the measured values of the output nodes.

In [ ]:
data_example_file = 'files/experiments_example.csv'
measured_nodes, perturbed_nodes = KGBN.extract_experiment_nodes(data_example_file)

In [ ]:
KGBN.vis_network(network, measured_nodes=measured_nodes, perturbed_nodes=perturbed_nodes)

In [ ]:
# Alternatively, an interactive visualization
KGBN.vis_network(network, output_html="files/ExampleBN.html", interactive=True, measured_nodes=measured_nodes, perturbed_nodes=perturbed_nodes)

## Compress the network

Before simulation, we can simplify the network by:
1. Removing non-observable nodes: 
2. Removing non-controllable nodes
3. Collapsing linear paths

This is because these nodes and edges do not affect the dynamics of the network for the given experiments, and we could simplify the network for better efficiency.

In [ ]:
compressed_network, compression_info = KGBN.compress_model(
    network,
    measured_nodes=measured_nodes,
    perturbed_nodes=perturbed_nodes
)

In [ ]:
compressed_network_string = '\n'.join(compressed_network.equations)
print("=========Compressed Network=========")
print(compressed_network_string)

In [ ]:
# visualize the compression difference
KGBN.vis_compression(
    network,
    compressed_network, 
    compression_info,
    output_html="files/compression.html",
    interactive=True
)

## Extend the network

Because the input model may not cover the genes we want to analyze, e.g., missing regulators or drug targets, we can extend the network using the knowledge graph (KG).  

Here, assume that we want to add `PDPK1` to the network to evaluate its inhibition effect.

**Approach of KG extension**:  
Given a list of genes of interest:

1. Find the Steiner subgraph for the given list of genes.
2. For each node in the subgraph, find all if its input nodes (i.e. all edges leading into that node).
3. For each such node, there is an activating relation if the edge is "up-regulates", and a repressing relation if the edge is "down-regulates".
4. Combine all of the edges with a "joiner function" - AND, OR, inhibitor wins, etc.
(Threshold value, majority voting, etc?)

In [ ]:
# First we need to specify the genes to be considered
# we will first get the genes from the compressed network
genes = compressed_network.nodeDict.keys()
# then we can add the genes of interest to the list
genes_of_interest = ['PDPK1']
genes = list(set(genes).union(set(genes_of_interest)))
print(genes)

In [ ]:
# get the knowledge graph-derived network
KG_string, relations = KGBN.load_signor_network(genes, joiner='inhibitor_wins')
print("\n")
print("=========Knowledge Graph Network=========")
print(KG_string)

**We can now merge the knowledge graph network with the compressed network.**   
  
There are four different ways to merge the networks:

1. AND
2. OR
3. Inhibitor Wins
4. PBN

For details, please refer to [LM-Merger](https://doi.org/10.1186/s12859-025-06212-2).

In [ ]:
KG = KGBN.load_network_from_string(KG_string)
# Merge the networks using inhibitor wins
extended_network_string = KGBN.merge_networks([compressed_network, KG], method="Inhibitor Wins", descriptive=True)

In [ ]:
# Merge the networks using PBN
# Here, a probability of 0.9 is used for rules from the original network
pbn_string = KGBN.merge_networks([compressed_network, KG], method="PBN", prob=0.9)
print("=========Merged PBN=========")
print(pbn_string)

Alternatively, instead of merging the two networks with all the rules, we can just extend the network with several new rules from the knowledge graph.   
This can be done through the `extend_networks` function: 

`extend_networks(original_network, new_network, nodes_to_extend, prob=0.5, descriptive=True)`

This will return a PBN where:
 - rules for the nodes_to_extend are added to the original network with a probability of prob.
 - rules for the nodes_to_extend from the original network will have a probability of 1-prob.
   - in case the new rules for nodes_to_extend include new nodes, they will also be added with a probability of 1.
   - The direct targets of nodes_to_extend are also extended with both rules and probabilities.
 - The rest of the rules are kept the same with a probability of 1.

In [ ]:
# Extend the network using PBN
extended_network_string = KGBN.extend_networks(compressed_network, KG, ['PDPK1'], prob=0.5)
print("\n")
print("=========Extended PBN=========")
print(extended_network_string)

In [ ]:
pbn = KGBN.load_pbn_from_string(extended_network_string)
KGBN.vis_network(pbn, output_html="files/ExtendedPBN.html", interactive=True, measured_nodes=measured_nodes, perturbed_nodes=perturbed_nodes)

## Optimize the network

We can now optimize the network using the experimental data by finding the best probabilities for each rule.  
This is done by minimizing the difference (MSE) between the predicted and experimental values of the output nodes in steady state.  

### Test for convergence

Before running the optimization, we can test how many Monte Carlo steps are needed to achieve a steady state. This can then be used to set the configuration for the optimization.

In [ ]:
# initialize the steady state calculator
calc = KGBN.SteadyStateCalculator(pbn)
# set the experimental conditions
calc.set_experimental_conditions(stimuli=['EGFR'],inhibitors=['PIK3CA'])
# compute the steady state
steady_state,convergence_info = calc.compute_steady_state(
    method='monte_carlo',
    n_runs=3,
    n_steps=2000,
    p_noise=0.05,
    analyze_convergence=True,
    output_node='CDK2')
print(steady_state)

### Run optimization

In [ ]:
# Configure optimizer
config = {
    'pso_params': {
        'n_particles': 100,
        'iters': 100,
        'options': {
            'c1': 0.5, # Cognitive parameter
            'c2': 0.3, # Social parameter
            'w': 0.9 # Inertia weight
        },
        'ftol': -1, # disable early stopping
        'ftol_iter': 5
    },
    'steady_state': {
        'method': 'monte_carlo',
            'monte_carlo_params': {
                'n_runs': 3,
                'n_steps': 2000,
                'p_noise': 0.05
            }
    },
    'max_try': 2,  # Maximum number of attempts if optimization fails
    'success_threshold': 0.02, # threshold for accepting fit,
    'display_rules_every': 0  # Display optimized rules every n iterations (0 = disabled)
}

In [ ]:
optimizer = KGBN.ParameterOptimizer(
    pbn,
    data_example_file, 
    config=config, 
    nodes_to_optimize=["AKT1","MAP2K1"],  # optimize the affected nodes
    verbose=False)

# Run optimization
result = optimizer.optimize(method='particle_swarm')

In [ ]:
optimizer.plot_optimization_history(result)

In [ ]:
# Configure optimizer
config = {
    'de_params': {
        'strategy': 'best1bin',
        'maxiter': 100,
        'popsize': 30,
        'tol': 0.0001, 
        'atol': 0,
        'mutation': (0.5, 1),
        'recombination': 0.7,
        'seed': 99,
        'disp': False,
        'init': 'sobol',
        'updating': 'deferred',
        'workers': -1,
        'early_stopping': False
    },
    'steady_state': {
        'method': 'monte_carlo',
            'monte_carlo_params': {
                'n_runs': 3,
                'n_steps': 2000,
                'p_noise': 0.05
            }
    },
    'max_try': 2,  # Maximum number of attempts if optimization fails
    'success_threshold': 0.01, # threshold for accepting fit,
    'display_rules_every': 0  # Display optimized rules every n iterations (0 = disabled)
}

optimizer_de = KGBN.ParameterOptimizer(
    pbn,
    data_example_file, 
    config=config, 
    nodes_to_optimize=["AKT1","MAP2K1"], 
    verbose=False)

# Run optimization
result_de = optimizer_de.optimize(method='differential_evolution')

In [ ]:
optimizer_de.plot_optimization_history(result_de)

## Evaluate the optimized network

In [ ]:
evaluator = KGBN.evaluate_optimization_result(
    result, 
    optimizer, 
    output_dir="files/PBN_evaluation_results",
    save=True,
    detailed=True
)

## Simulate the optimized network

In [ ]:
pbn_optimized = optimizer.get_optimized_pbn(result)
KGBN.vis_network(pbn_optimized,"files/OptimizedPBN.html",interactive=True,measured_nodes=measured_nodes,perturbed_nodes=perturbed_nodes)

In [ ]:
import pandas as pd
import numpy as np
import itertools

perturbation_genes = ['PIK3CA','PDPK1','MAP2K1','MDM2']
stimuli_genes = ['EGF','DNA_DAMAGE']
pertubations = []
for r in range(1, len(perturbation_genes)+1):
    for combo in itertools.combinations(perturbation_genes, r):
        pertubations.append(','.join(combo))
stimuli = []
for r in range(1, len(stimuli_genes)+1):
    for combo in itertools.combinations(stimuli_genes, r):
        stimuli.append(','.join(combo))

df_results = pd.DataFrame()
for stimulus in stimuli:
    for pertubation in pertubations:
        calc = SteadyStateCalculator(pbn_optimized)
        calc.set_experimental_conditions(stimuli=[stimulus],inhibitors=[pertubation])
        steady_state = calc.compute_steady_state(
            method='monte_carlo',
            n_runs=3,
            n_steps=1000,
            p_noise=0.05)
        conditions = [f"Stimulus:{stimulus}; Inhibitor:{pertubation}"]
        df_steady_state = pd.DataFrame(steady_state,index=pbn_optimized.nodeDict.keys(),columns=conditions).T
        df_results = pd.concat([df_results,df_steady_state])
df_results

In [ ]:
import re
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

response_genes     = ['TP53', 'AKT1']

# build binary matrix from index strings
conditions = df_results.index.astype(str)
stimuli_block  = pd.DataFrame(0, index=stimuli_genes,      columns=conditions, dtype=int)
perturb_block  = pd.DataFrame(0, index=perturbation_genes, columns=conditions, dtype=int)
for cond in conditions:
    m_stim = re.search(r"Stimulus:([^;]+)", cond)
    m_inhib = re.search(r"Inhibitor:([^;]+)", cond)
    if m_stim:
        for g in [x.strip() for x in m_stim.group(1).split(',') if x.strip().lower() not in ['none', '']]:
            if g in stimuli_block.index:        stimuli_block.loc[g, cond] = 1
    if m_inhib:
        for g in [x.strip() for x in m_inhib.group(1).split(',') if x.strip().lower() not in ['none', '']]:
            if g in perturb_block.index:        perturb_block.loc[g, cond] = 1

response_block = df_results[response_genes].T

bw   = ListedColormap(['white', 'black'])
heat = sns.color_palette('coolwarm', as_cmap=True)
vmin, vmax = response_block.values.min(), response_block.values.max()

heights = [len(stimuli_block), len(perturb_block), len(response_block)]
fig, (ax_stim, ax_pert, ax_resp) = plt.subplots(
    3, 1,
    figsize=(0.5 * df_results.shape[0], 0.5 * sum(heights)),
    sharex=True,
    gridspec_kw={'height_ratios': heights}
)

sns.heatmap(stimuli_block, cmap=bw, vmin=0, vmax=1, ax=ax_stim,
            cbar=False, linewidths=.5, linecolor='grey')
ax_stim.set_title('Stimuli (black = active)')
ax_stim.set_ylabel('')
ax_stim.set_xticks([])
ax_stim.set_yticklabels(ax_stim.get_yticklabels(), rotation=0)

sns.heatmap(perturb_block, cmap=bw, vmin=0, vmax=1, ax=ax_pert,
            cbar=False, linewidths=.5, linecolor='grey')
ax_pert.set_title('Perturbations (black = KO)')
ax_pert.set_ylabel('')
ax_pert.set_xticks([])
ax_pert.set_yticklabels(ax_pert.get_yticklabels(), rotation=0)

sns.heatmap(response_block, cmap=heat, vmin=vmin, vmax=vmax, ax=ax_resp,
            cbar=False, linewidths=.5, linecolor='white', annot=True, fmt='.1f')
ax_resp.set_title('TP53 and AKT1 steady states')
ax_resp.set_ylabel('')
ax_resp.set_xlabel('')
ax_resp.set_xticks([])
ax_resp.set_yticklabels(ax_resp.get_yticklabels(), rotation=0)

for ax in (ax_stim, ax_pert, ax_resp):
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.show()

# Generate synthetic data

In [ ]:
import sys
import pandas as pd
import numpy as np
import KGBN
 

In [ ]:
pbn_hypothesis = KGBN.load_pbn_from_string(
"""
AKT1 = !CASP3 & !PTEN & ( PDPK1 | PIK3CA ), 0.7
AKT1 = PIK3CA & !PTEN, 0.3
CASP3 = !AKT1 & TP53, 1.0
CDK2 = !TP53 & ( AKT1 | MAPK1 ), 1.0
DNA_DAMAGE = DNA_DAMAGE, 1.0
EGF = EGF, 1.0
EGFR = EGF, 1.0
MAP2K1 = !MAPK1 & ( MAP2K1 | PDPK1 ), 0.4
MAP2K1 = EGFR, 0.6
MAPK1 = MAP2K1, 1.0
MDM2 = AKT1, 1.0
PDPK1 = PDPK1, 1.0
PIK3CA = EGFR | MAPK1, 1.0
PTEN = !PIK3CA, 1.0
TP53 = DNA_DAMAGE & !MDM2, 1.0
"""
)
config = {
    'steady_state': {
        'method': 'monte_carlo',
            'monte_carlo_params': {
                'n_runs': 3,
                'n_steps': 2000,
                'p_noise': 0.05
            }
    }
}
KGBN.generate_experiments(pbn_hypothesis,'files/experiments_blank.csv',config, 'files/experiments_example.csv', 2)